# 금리–서울 아파트 거래량 시계열 분석 (수정본)

이 노트북은 기존 분석의 다음 문제를 바로잡고 보완한 버전입니다.

**수정한 치명적 오류**
1. CCSI 데이터를 실제로 로드 (기존엔 `ccsi` 미정의로 실행 불가)
2. Granger 검정을 올바른 변수쌍으로 (기존엔 금리↔금리, 거래량 빠짐)
3. 누락됐던 OLS 회귀(Model 1/2/3) 코드 복원

**보완한 방법론**
4. OLS 잔차 자기상관 진단 + HAC(Newey-West) 강건 표준오차
5. 공적분(Engle-Granger) 검정 → 차분 VAR 정당화 여부 확인
6. 계절성 통제(월 더미) 회귀
7. VAR 사후진단(안정성·자기상관) + IRF/FEVD
8. 정책 더미를 VAR 외생변수(exog)로 처리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, coint
from statsmodels.tsa.api import VAR
import statsmodels.api as sm

## 1. 데이터 로드 및 병합 (CCSI 포함 — 수정)

In [ ]:
df = pd.read_csv('merged_data.csv')
df['date'] = pd.to_datetime(df['date'])

ccsi = pd.read_csv('ccsi.csv')        # ★ 기존에 빠져 있던 부분
ccsi['date'] = pd.to_datetime(ccsi['date'])

df = pd.merge(df, ccsi, on='date', how='left')
print('관측치:', len(df), '| CCSI 결측:', df['ccsi'].isna().sum())
df.head()

In [ ]:
# 차분 변수
for c in ['base_rate', 'mortgage_rate', 'trade_volume', 'ccsi']:
    df[c + '_diff'] = df[c].diff()

# 정책 더미
df['dsr_dummy'] = (df['date'] >= '2022-01-01').astype(int)
df['ltv_dummy'] = (df['date'] >= '2017-08-01').astype(int)

## 2. 상관분석 (원자료 / 차분)

In [ ]:
print('원자료 상관'); print(df[['base_rate','mortgage_rate','trade_volume','ccsi']].corr().round(3))
print('\n차분 상관'); print(df[['base_rate_diff','mortgage_rate_diff','trade_volume_diff','ccsi_diff']].corr().round(3))

## 3. ADF 정상성 검정

주의: ADF는 시차 선택에 민감합니다. 거래량·CCSI는 자동 시차가 12로 잡히는 경향이 있는데,
이는 **연 단위 계절성**의 신호입니다(섹션 7에서 통제).

In [ ]:
def adf_test(series, name):
    r = adfuller(series.dropna(), autolag='AIC')
    print(f'{name:18s} ADF={r[0]:7.3f}  p={r[1]:.4f}  lag={r[2]:2d}  -> ' +
          ('정상' if r[1] < 0.05 else '비정상'))

print('[원자료]')
for c in ['base_rate','mortgage_rate','trade_volume','ccsi']:
    adf_test(df[c], c)
print('\n[1차 차분]')
for c in ['base_rate','mortgage_rate','trade_volume','ccsi']:
    adf_test(df[c+'_diff'], c+'_diff')

## 4. 공적분 검정 (Engle-Granger)

기준금리·주담대금리가 모두 I(1)이므로, 둘이 공적분되어 있으면 VECM이 맞고
공적분이 없으면 차분 VAR가 타당합니다. → 우리 분석 접근의 정당성 점검.

In [ ]:
score, pval, _ = coint(df['mortgage_rate'], df['base_rate'])
print(f'Engle-Granger 통계량={score:.3f}  p={pval:.4f}  -> ' +
      ('공적분 있음 → VECM 고려' if pval < 0.05 else '공적분 없음 → 차분 VAR 타당'))

## 5. Granger 인과성 — 올바른 방향 (주담대금리변화 → 거래량변화)

In [ ]:
# ★ 기존 오류: df[['mortgage_rate_diff','base_rate_diff']] 였음(거래량 없음)
# 첫 열 = 결과(피설명), 둘째 열 = 원인(설명) 순서
g = df[['trade_volume_diff', 'mortgage_rate_diff']].dropna()
res = grangercausalitytests(g, maxlag=6, verbose=True)

## 6. OLS 회귀 — 차분 기준 (★ 헤드라인 모형)

**왜 차분 회귀를 본 모형으로 쓰는가:**
원자료 수준 회귀(아래 6-1 참고용)는 잔차 자기상관이 심해 Durbin-Watson ≈ 0.77로,
교수님이 경고한 가짜회귀 패턴(낮은 DW + 그럴듯한 R²)을 그대로 보입니다.
종속·설명변수를 모두 차분하면 정상성이 확보되고 자기상관이 사라져(DW ≈ 2.4)
통계적으로 신뢰할 수 있는 추정이 됩니다. 대신 설명력(Adj R²)은 낮아집니다 —
월별 변화량 회귀에서는 정상적인 현상입니다.

In [ ]:
def run_ols(y, cols, label, data=None):
    sub = (data if data is not None else df).dropna(subset=[y] + cols)
    X = sm.add_constant(sub[cols])
    m_ols = sm.OLS(sub[y], X).fit()
    m_hac = sm.OLS(sub[y], X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    print(f'\n[{label}] n={int(m_ols.nobs)}  R²={m_ols.rsquared:.3f}  '
          f'Adj.R²={m_ols.rsquared_adj:.3f}  DW={sm.stats.durbin_watson(m_ols.resid):.3f}')
    print(pd.DataFrame({'coef': m_ols.params.round(1),
                        'p(OLS)': m_ols.pvalues.round(4),
                        'p(HAC)': m_hac.pvalues.round(4)}))

# 본 모형: 종속변수 = 거래량 변화(차분)
run_ols('trade_volume_diff', ['mortgage_rate_diff'], 'Model 1 (금리만)')
run_ols('trade_volume_diff', ['mortgage_rate_diff','ccsi_diff'], 'Model 2 (+심리)')

**해석:** 차분 기준에서 주담대금리 변화는 거래량 변화에 유의한 음(-)의 효과를
가집니다(HAC p ≈ 0.001). 반면 CCSI 변화(ccsi_diff)는 유의하지 않습니다(p ≈ 0.5).
즉 기존 보고서의 'CCSI 유의'는 수준 회귀의 자기상관에서 비롯된 착시였습니다.

> 참고: 정책 더미를 차분하면 변경 시점 1개월짜리 펄스가 되어 단일 이상치를
> 잡는 것에 불과하므로, 차분 회귀에서는 정책 더미를 제외합니다.

### 6-1. (참고) 수준 회귀의 자기상관 문제 — 왜 차분을 쓰는지 보여주는 비교

In [ ]:
# 비교용: 기존 방식인 원자료 수준 회귀. DW가 매우 낮음(자기상관 심각)에 주목.
run_ols('trade_volume', ['mortgage_rate','ccsi'], '(참고) 수준 회귀 Model 2')

## 7. 계절성 통제 (차분 회귀 + 월 더미)

월별 거래량은 봄·가을 성수기 등 연 단위 계절성이 강합니다(섹션 3의 lag=12 신호).
차분 모형에 월 더미를 더하면 설명력이 개선되는지, 금리 효과가 유지되는지 확인합니다.

In [ ]:
df['month'] = df['date'].dt.month
month_d = pd.get_dummies(df['month'], prefix='m', drop_first=True).astype(float)
cols = ['mortgage_rate_diff','ccsi_diff']
sub = df.dropna(subset=['trade_volume_diff'] + cols)
X = sm.add_constant(pd.concat([sub[cols], month_d.loc[sub.index]], axis=1))
m = sm.OLS(sub['trade_volume_diff'], X).fit(cov_type='HAC', cov_kwds={'maxlags':4})
print(f'Adj.R²={m.rsquared_adj:.3f}  DW={sm.stats.durbin_watson(m.resid):.3f}  '
      '(계절성 통제 시 설명력 개선)')
print(m.params[cols].round(1)); print(m.pvalues[cols].round(4))

## 8. VAR + 사후진단 + IRF/FEVD (정책더미는 exog로 처리)

In [ ]:
endog = df[['base_rate_diff','mortgage_rate_diff','trade_volume_diff','ccsi_diff']].dropna()
exog = df.loc[endog.index, ['dsr_dummy','ltv_dummy']]   # ★ 더미는 외생변수로

model = VAR(endog, exog=exog)
print(model.select_order(maxlags=6).summary())

In [ ]:
var_res = model.fit(maxlags=2, ic=None)   # 정보기준 결과 보고 시차 조정
print('모형 안정성(stable):', var_res.is_stable())   # 모든 근이 단위원 안쪽이어야 함

# 잔차 자기상관 검정 (Ljung-Box / Portmanteau)
print(var_res.test_whiteness(nlags=12).summary())

In [ ]:
# 거래량의 금리 충격 반응 (IRF) — VAR의 핵심 산출물
irf = var_res.irf(12)
irf.plot(impulse='mortgage_rate_diff', response='trade_volume_diff')
plt.tight_layout(); plt.show()

# 분산분해: 거래량 변동을 각 변수가 몇 % 설명하는가
var_res.fevd(12).plot()
plt.tight_layout(); plt.show()

In [ ]:
# VAR 기반 Granger 인과성 (금리변화 -> 거래량변화)
print(var_res.test_causality('trade_volume_diff', ['mortgage_rate_diff'], kind='f').summary())

## 9. 해석 요약 (차분 기준, 통계적으로 엄밀한 버전)

- **공적분 없음** → 차분 VAR/차분 OLS 접근이 타당함.
- **헤드라인 = 차분 회귀.** 주담대금리 변화는 거래량 변화에 유의한 음(-)의 효과
  (HAC p ≈ 0.001~0.02, DW ≈ 2.4로 자기상관 없음). 단 Adj.R²는 0.03~0.10으로 낮음
  — 월별 변화량 회귀에서는 정상적인 수준.
- **CCSI는 차분 기준에서 유의하지 않음.** 기존 보고서의 'CCSI 유의'는 수준 회귀의
  자기상관(DW≈0.77)에서 비롯된 착시였음 → 결론 재서술 필요.
- **계절성 통제 시 설명력 개선**(Adj.R² 약 0.03→0.10), 금리 효과는 유지.
- **Granger·VAR 모두 금리→거래량 선행효과는 5%에서 비유의** → 기존 결론과 일치.
- **정책 더미는 본 모형에서 제외/비유의** → 단순 0/1(또는 펄스) 더미의 한계.

**한 줄 결론:** 설명력은 낮지만, 시계열 정상성·자기상관을 엄밀히 통제했을 때
견고하게 남는 것은 '주담대금리 변화 → 거래량 변화의 음(-)의 관계' 하나이며,
소비심리·정책 더미의 독립적 효과는 통계적으로 확정하기 어렵다.